# We will explor a few things
- What is a window
- Fixed-size window vs time-based window
- Moving context concept
- Rolling vs aggregation
- Rolling vs expanding vs Shifting


1. Small moving partition, subset of observation used to calculate statistics
2. Window work on 2 modes , number(fixed-size) and time frame
```python
rolling(window=7) # this 7 means consecutive backword 7 data
rolling('7min') # This 7min means consecutive backword 7 min data
```
3. Moving context :  Moving backward by default by we can make it forward
```python
rolling() #backward by default
rolling(center=True) # Forward now
```
4. Rolling aggregation means it will work using groupby and aggregation method
```python
groupby('month').sum()
```
5. Rolling vs Expanding vs Shifting
```text
Rolling we understand correctly that it moves window in specific steps and offer to apply statistics calculation
Expanding offer cucumulation calculation
Shifting add more padding and we will be able to see something like previous data on current row 
```


## Core Rolling Api

- Understanding Series.rolling()
- Understanding DataFrame.rolling()
- Rolling object lifecycle
- Syntax patterns
- Default behavior
- Return objects
- How rolling connects with aggregation function 

In [2]:
# Series rolling example:
import  pandas as pd
from triton.language import cumsum

sales =pd.Series([100,120,150,130,170],
                 index =['Mon','Tue','Wed','Thu','Fri'])
rolling_average =sales.rolling(window=3).mean()
print(rolling_average)

Mon           NaN
Tue           NaN
Wed    123.333333
Thu    133.333333
Fri    150.000000
dtype: float64


In [5]:
# DataFrame rolling example

df =pd.DataFrame({'sales':sales})
# print(df)
# print('__________________________')
df['mean']=df['sales'].rolling(window=3).mean()
print(df)


     sales
Mon    100
Tue    120
Wed    150
Thu    130
Fri    170
__________________________
     sales        mean
Mon    100         NaN
Tue    120         NaN
Wed    150  123.333333
Thu    130  133.333333
Fri    170  150.000000


### What does center=True/False means
```python
input =[10,20,30,40,50]
center=False
```
chunk will be something like that for mean()
```python
  [NaN,NaN,10], #NaN
  [NaN,10,20],  #20
  [10,20,30],   #20
  [20,30,40],   #30
  [30,40,50]    #40
```

**What happen when center will be True**
```python
  [NaN,10,20], #NaN  -> Center 10
  [10,20,30],  #20
  [20,30,40],  #30
  [30,40,50],  #30
  [40,50,NaN]  #40
```

In [8]:
s =pd.Series([10,20,30,40,50])
centerT=s.rolling(window=3,center=True).mean()
print(centerT)

0     NaN
1    20.0
2    30.0
3    40.0
4     NaN
dtype: float64


### Rolling object lifecycle
it's divided in 3 stage
- input a normal variable could be either series or DataFrame
- Create window definition by using rolling() just after Series or DataFrame
- Apply aggregation method just after rolling
```text
        - mean
        - sum
        - median
```

### Syntax pattern
```python
sales.rolling(window=2,
min_periods=None,
center=False,
closed=None
).aggregate_function()

```
### Default behaviour
- Backword statistical calculation
- Default center=false
- Default `min_periods` exactly as windows number for integer for datetime it's 1
- Window moves one row at a time, step=1
- Default `closed=None`

## Deep drive in side `closed=left/right/both/neither/None`
```txt
left boundary---------------------------- right boundary
10:00 ------------------------------------ 10:10
```

| closed  | include left | include right |
|:-------:|:------------:|:-------------:|
|  right  |   &cross;    |    &check;    |
|  left   |   &check;    |    &cross;    |
|  both   |   &check;    |    &check;    |
| neither |   &cross;    |    &cross;    |




# Internal Mechanics
- Index based windows
- Position based windows


## Answer 
- Index based window
  > In this case window can move it's frame with certain column counts example
```python 
rolling(7)
```
always frame will move 7 columns 

- Position based window
  > In this case window will move uncertain number of columns , we don't know about it, it's respect to the data time
  ```python 
  rolling('7d')
  ```
  frame will move untill 7 days , we don't know exact column in this 7 days , it's completely unknown columns number

 # Custom rolling logic
  - `rolling().apply()`
  - Writing custom function
  - Function input behavior
  - Lambda function with rolling
  - Returning scalar values
  - Complex calculation inside windows
  - Custom statistical measures

In [18]:
df =pd.DataFrame({
    'day':pd.date_range('2026-01-01', periods=8),
    'sales':[100,120,90,150,200,170,220,250],
    'customer':[10,12,9,15,20,17,22,25]
})


'''
@Input pd.Series
@output list float scaler
output pd.Series is deprecated
'''
def add_one_more_ten(x:pd.Series)->list[float]:
    return x.iloc[0]+10


df['t_sale']=df.rolling(window=1)['sales'].apply(lambda x: x.iloc[0]+10)
df['t2_sale']=df.rolling(window=1)['sales'].apply(add_one_more_ten)
print(df)

         day  sales  customer  t_sale  t2_sale
0 2026-01-01    100        10   110.0    110.0
1 2026-01-02    120        12   130.0    130.0
2 2026-01-03     90         9   100.0    100.0
3 2026-01-04    150        15   160.0    160.0
4 2026-01-05    200        20   210.0    210.0
5 2026-01-06    170        17   180.0    180.0
6 2026-01-07    220        22   230.0    230.0
7 2026-01-08    250        25   260.0    260.0


# Rolling feature Engineering for Machine learning
- Lag Features
- Moving averages
- Rolling statistics
- Trend features
- Preventing data leakage
